In [1]:
# 📦 IMPORT ANALYSIS FUNCTIONS FROM MODULE
import affine_adversarial_attack_analysis as analysis_funcs

# Import all the analysis functions
from affine_adversarial_attack_analysis import (
    random_sample_attack_data,
    generate_non_digit_images,
    analyze_non_digit_reconstructions,
    find_optimal_inputs_for_coordinates,
    explore_coordinate_space,
    targeted_content_attack,
    targeted_transform_attack,
    evaluate_adversarial_training_benefit,
    create_synthetic_attack_targets,
    statistical_attack_analysis,
    generate_attack_report,
    manual_reparameterize,
    access_inner_autoencoder_components,
    fixed_latent_space_attack
)


In [2]:
# 📦 IMPORTS & SETUP
import torch
import matplotlib.pyplot as plt
import numpy as np
import random

# Affine autoencoder modules
import affine_autoencoder_shared as shared
import structured_2d6d_autoencoder as s2d6d
from affine_adversarial_attacks import *

print(f"✅ Imports complete - PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

✅ All imports successful!
PyTorch: 2.7.1, CUDA: False


In [3]:
# ⚙️ CONFIGURATION
CONFIG = {
    'content_latent_dim': 2,
    'transform_latent_dim': 6, 
    'total_latent_dim': 8,
    'batch_size_test': 64,
    'data_dir': '../data',
    'n_attack_samples': 8,
    'attack_epsilons': [0.05, 0.1, 0.15, 0.2],
    'alpha': 1.0,
    'beta': 0.01,
    'mixed_precision': False,
    'device_preference': 'cuda'
}

print(f"✅ Config: {CONFIG['total_latent_dim']}D VAE, {CONFIG['n_attack_samples']} samples")

🎯 AFFINE AUTOENCODER ATTACK CONFIGURATION:
  Model: 8D unified VAE
  Attack samples: 8
  Attack epsilons: [0.05, 0.1, 0.15, 0.2]
  Beta (KL weight): 0.01


In [4]:
# 🖥️ SETUP DEVICE & DATA
try:
    device = shared.get_cloud_device(CONFIG)
except:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# Data loading with fallbacks
train_loader = test_loader = None
try:
    train_loader, test_loader = shared.get_cloud_mnist_loaders(
        batch_size_test=CONFIG['batch_size_test'], data_dir=CONFIG['data_dir']
    )
except:
    try:
        from torchvision import datasets, transforms
        from torch.utils.data import DataLoader
        import os
        os.makedirs(CONFIG['data_dir'], exist_ok=True)
        
        transform = transforms.Compose([
            transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))
        ])
        test_dataset = datasets.MNIST(root=CONFIG['data_dir'], train=False, download=True, transform=transform)
        test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size_test'], shuffle=False)
    except:
        # Synthetic fallback
        from torch.utils.data import TensorDataset, DataLoader
        synthetic_images = torch.randn(100, 1, 28, 28)
        synthetic_labels = torch.randint(0, 10, (100,))
        test_loader = DataLoader(TensorDataset(synthetic_images, synthetic_labels), 
                                batch_size=CONFIG['batch_size_test'], shuffle=False)
        print("⚠️ Using synthetic data")

print(f"✅ Data ready: {len(test_loader)} batches")

🖥️ Setting up device and data loaders...
📱 Testing device setup...
🍎 Apple MPS device
✅ Device configured: mps

📊 Setting up data loaders...
⏳ Attempting to load MNIST data (this may take time for first download)...
📁 Data directory: ../data
📊 Train batches: 235, Test batches: 157
✅ Data loaders created successfully!
📊 Test loader: 157 batches of 64

🎯 Final setup:
  Device: mps
  Test loader ready: ✅
  Batches available: 157
  Batch size: 64


In [14]:
# 🔧 LOAD MODEL
model_file = "unified_8d_vae_unified_8d_vae_20250724_150249.pth"

checkpoint = torch.load(model_file, map_location=device, weights_only=False)
model = s2d6d.StructuredAffineInvariantAutoEncoder(
    content_dim=CONFIG['content_latent_dim'], 
    transform_dim=CONFIG['transform_latent_dim']
)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()

print(f"✅ Model loaded: Content({CONFIG['content_latent_dim']}) + Transform({CONFIG['transform_latent_dim']})")


✅ Loaded model: unified_8d_vae_unified_8d_vae_20250724_150249.pth
Model architecture: Content(2) + Transform(6) = 8D
Device: mps
Model config: {'content_latent_dim': 2, 'transform_latent_dim': 6, 'total_latent_dim': 8, 'epochs': 60, 'learning_rate': 0.001, 'batch_size_train': 256, 'batch_size_test': 128, 'alpha': 1.2, 'beta': 0.01, 'force_cuda': True, 'mixed_precision': True, 'gradient_clip': 1.0, 'pin_memory': True, 'num_workers': 4, 'weight_decay': 1e-05, 'lr_scheduler': True, 'early_stopping': True, 'patience': 15, 'data_dir': '../data', 'save_dir': './', 'checkpoint_freq': 10}


In [7]:
# 🎯 ATTACK SETUP
test_iter = iter(test_loader)
test_images, test_labels = next(test_iter)

batch_size = test_images.size(0)
n_samples = CONFIG.get('n_attack_samples', 8)
random_indices = random.sample(range(batch_size), min(n_samples, batch_size))

attack_samples = test_images[random_indices].to(device)
attack_labels = test_labels[random_indices]

attacker = AffineAdversarialAttacks(model, device)
attack_results = {}

print(f"✅ Attack setup: {len(attack_samples)} samples, epsilons {CONFIG['attack_epsilons']}")

🎯 Starting Modular Affine Autoencoder Attack Analysis...
This analysis is broken into smaller pieces for better performance and progress tracking.

📊 Selected 8 random samples for attack analysis
🎯 Attack epsilons to test: [0.05, 0.1, 0.15, 0.2]
🎪 Attack methods: FGSM, PGD, Latent Space

✅ Setup completed! Ready to run individual attack methods...


In [9]:
# 🔧 DEVICE COMPATIBILITY CHECK
if str(device) == 'mps':
    try:
        # Test MPS grid sampler compatibility
        test_img = torch.randn(1, 1, 28, 28, requires_grad=True).to(device)
        test_theta = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], requires_grad=True).unsqueeze(0).to(device)
        grid = F.affine_grid(test_theta, test_img.size(), align_corners=False)
        result = F.grid_sample(test_img, grid, align_corners=False)
        result.mean().backward()
        
        attack_device = device
        model_cpu = model
        attack_samples_cpu = attack_samples
        attack_labels_cpu = attack_labels
    except NotImplementedError:
        print("⚠️ MPS fallback to CPU for attacks")
        attack_device = torch.device('cpu')
        model_cpu = model.cpu()
        attack_samples_cpu = attack_samples.cpu()
        attack_labels_cpu = attack_labels
else:
    attack_device = device
    model_cpu = model
    attack_samples_cpu = attack_samples
    attack_labels_cpu = attack_labels

attacker = AffineAdversarialAttacks(model_cpu, attack_device)
print(f"✅ Attack device: {attack_device}")

🔧 Checking device compatibility for adversarial attacks...
⚠️  MPS device detected - testing grid sampler compatibility...
❌ MPS device doesn't support grid_sampler_2d_backward
🔄 Falling back to CPU for adversarial attacks...
✅ Fallback to CPU configured
🎯 Attack device: cpu
📊 Model device: cpu
📊 Samples device: cpu
✅ Device compatibility check completed!


In [10]:
# 🎲 RANDOM SAMPLING
ENABLE_RANDOM_SAMPLING = True
RANDOM_SAMPLE_SIZE = 4
RANDOM_SEED = 42

if ENABLE_RANDOM_SAMPLING:
    original_size = len(attack_samples_cpu)
    attack_samples_cpu, attack_labels_cpu = random_sample_attack_data(
        attack_samples_cpu, attack_labels_cpu, 
        n_random_samples=RANDOM_SAMPLE_SIZE, 
        seed=RANDOM_SEED
    )
    print(f"✅ Sampled {original_size} → {len(attack_samples_cpu)} samples, labels: {attack_labels_cpu.tolist()}")
else:
    print(f"✅ Using all {len(attack_samples_cpu)} samples")

🎲 Random Sampling Options for Attack Data
🎯 Random sampling enabled: 4 samples
🎲 Randomly sampling 4 from 8 available attack samples...
✅ Random sampling completed!
   Original shape: torch.Size([8, 1, 28, 28])
   Sampled shape: torch.Size([4, 1, 28, 28])
   Random indices: [1, 0, 5, 2]
📊 Attack data updated:
   Samples: 8 → 4
   Selected labels: [5, 9, 6, 1]

✅ Final attack data ready:
   Samples shape: torch.Size([4, 1, 28, 28])
   Labels: [5, 9, 6, 1]
   Device: cpu


In [12]:
# 🔧 MODEL OUTPUT ANALYSIS
test_image = attack_samples_cpu[0:1]
with torch.no_grad():
    model_outputs = model_cpu(test_image)

print(f"Model outputs: {len(model_outputs)} tensors")
for i, output in enumerate(model_outputs):
    if output is not None:
        print(f"  [{i}]: {output.shape}, range [{output.min():.3f}, {output.max():.3f}]")
        if output.shape == test_image.shape:
            mse = torch.mean((output - test_image) ** 2).item()
            print(f"       MSE vs input: {mse:.6f}")

# Model output structure: [0] = complete_reconstruction, [1] = content_latents
print(f"✅ Using output[0] as reconstruction, output[1] as content latents")

🔧 DEBUGGING MODEL OUTPUT STRUCTURE


NameError: name 'orig_images' is not defined

In [ ]:
# 🚀 EXECUTE ATTACKS
print("🚀 Running adversarial attacks...")

# Run FGSM attacks
for eps in CONFIG['attack_epsilons']:
    try:
        adv_images = attacker.fgsm_attack(attack_samples_cpu, eps)
        attack_results[f'fgsm_eps_{eps}'] = {
            'adversarial_images': adv_images,
            'epsilon': eps,
            'success_rate': ((adv_images - attack_samples_cpu).abs().max() > 0.01).float().mean().item()
        }
        print(f"  FGSM ε={eps}: Success rate {attack_results[f'fgsm_eps_{eps}']['success_rate']:.3f}")
    except Exception as e:
        print(f"  FGSM ε={eps}: Failed - {str(e)[:50]}")

# Run PGD attacks  
for eps in CONFIG['attack_epsilons'][:2]:  # Limit PGD to first 2 epsilons
    try:
        adv_images = attacker.pgd_attack(attack_samples_cpu, eps, alpha=eps/5, steps=10)
        attack_results[f'pgd_eps_{eps}'] = {
            'adversarial_images': adv_images,
            'epsilon': eps,
            'success_rate': ((adv_images - attack_samples_cpu).abs().max() > 0.01).float().mean().item()
        }
        print(f"  PGD ε={eps}: Success rate {attack_results[f'pgd_eps_{eps}']['success_rate']:.3f}")
    except Exception as e:
        print(f"  PGD ε={eps}: Failed - {str(e)[:50]}")

# Latent space attack
try:
    adv_images = attacker.latent_space_attack(attack_samples_cpu, noise_scale=0.1)
    attack_results['latent_attack'] = {
        'adversarial_images': adv_images,
        'noise_scale': 0.1,
        'success_rate': ((adv_images - attack_samples_cpu).abs().max() > 0.01).float().mean().item()
    }
    print(f"  Latent: Success rate {attack_results['latent_attack']['success_rate']:.3f}")
except Exception as e:
    print(f"  Latent: Failed - {str(e)[:50]}")

print(f"✅ Attack results: {len(attack_results)} methods completed")